# Full Trenberth Diagram Calibration

This notebook calibrates **23 parameters** spanning shortwave radiation, longwave radiation, surface albedo, and land hydrology to simultaneously match the [Trenberth et al. (2009)](https://doi.org/10.1175/2008BAMS2634.1) global energy budget across **6 fluxes**:

| Flux | Target | Description |
|------|--------|-------------|
| OSR  | 101.9 W/m² | Outgoing shortwave at top-of-atmosphere |
| SRU  | 23.0 W/m²  | Surface shortwave reflected upward |
| SRD  | 168.0 W/m² | Surface shortwave absorbed downward |
| OLR  | 235.0 W/m² | Outgoing longwave at top-of-atmosphere |
| LRD  | 333.0 W/m² | Surface longwave downward (greenhouse back-radiation) |
| LRU  | 398.0 W/m² | Surface longwave upward (Stefan-Boltzmann emission) |

This is an extension of `radiation_sw.ipynb`: we add longwave emissivities and land-hydrology parameters. Including OLR in the loss couples the SW and LW optimisation — changing cloud albedo now also feeds back through the LW budget via surface temperature.

**Expected runtime:** ~8–12 hours at T31. Use Section 5 (quick test) first.

## 1. Setup

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using SpeedyCalibration
using Optimisers
using CairoMakie
using GeoMakie
using Dates
using Printf

## 2. Define Trainable Parameters

We train **23 parameters** across four modules. All paths and bounds are copied
directly from `full_trend_birth_training.ipynb`.

**`absorptivity_water_vapor` needs `grad_scale=0.01`.**
The physical gradient is ~200× weaker than the cloud parameters because it is
multiplied by specific humidity (q ≈ 0.005). Its lower bound is 60 (not 0) because
the model becomes numerically unstable below ~57.

**`snow_melting_threshold` needs `grad_scale=0.01`.**
Measured in Kelvin (~275 K), so a 1 W/m² flux change produces a ~100× smaller
raw gradient than dimensionless parameters.

**Parameters with zero gradient (excluded):**
- `conv_time_scale` — integer conversion in `Second(time_scale).value` is opaque to Enzyme
- `lsc_rh_threshold` — step-function boundary; gradient is zero almost everywhere


In [ ]:
param_specs = [
    # ── SW cloud reflection ───────────────────────────────────────────────────
    ParamSpec(:cloud_albedo,
        [:shortwave_radiation, :clouds, :cloud_albedo];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_cover_max,
        [:shortwave_radiation, :clouds, :stratocumulus_cover_max];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_albedo,
        [:shortwave_radiation, :clouds, :stratocumulus_albedo];
        bounds=(0.10f0, 0.90f0), initial=0.50f0),
    ParamSpec(:precipitation_weight,
        [:shortwave_radiation, :clouds, :precipitation_weight];
        bounds=(0.0f0, 0.8f0), initial=0.20f0),

    # ── SW atmospheric absorption ─────────────────────────────────────────────
    ParamSpec(:absorptivity_water_vapor,
        [:shortwave_radiation, :transmissivity, :absorptivity_water_vapor];
        bounds=(60f0, 140f0), initial=75f0, grad_scale=0.01f0),
    ParamSpec(:absorptivity_dry_air,
        [:shortwave_radiation, :transmissivity, :absorptivity_dry_air];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:absorptivity_aerosol,
        [:shortwave_radiation, :transmissivity, :absorptivity_aerosol];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:ozone_absorption,
        [:shortwave_radiation, :radiative_transfer, :ozone_absorption];
        bounds=(0.002f0, 0.020f0), initial=0.01f0),

    # ── Surface albedo ────────────────────────────────────────────────────────
    ParamSpec(:albedo_land,
        [:albedo, :land, :albedo_land];
        bounds=(0.10f0, 0.70f0), initial=0.40f0),
    ParamSpec(:albedo_high_vegetation,
        [:albedo, :land, :albedo_high_vegetation];
        bounds=(0.04f0, 0.26f0), initial=0.15f0),
    ParamSpec(:albedo_low_vegetation,
        [:albedo, :land, :albedo_low_vegetation];
        bounds=(0.05f0, 0.35f0), initial=0.20f0),
    ParamSpec(:albedo_snow,
        [:albedo, :land, :albedo_snow];
        bounds=(0.15f0, 0.75f0), initial=0.40f0),
    ParamSpec(:snow_depth_scale,
        [:albedo, :land, :snow_depth_scale];
        bounds=(0.005f0, 0.20f0), initial=0.05f0),
    ParamSpec(:albedo_ocean,
        [:albedo, :ocean, :albedo_ocean];
        bounds=(0.02f0, 0.10f0), initial=0.06f0),
    ParamSpec(:albedo_ice,
        [:albedo, :ocean, :albedo_ice];
        bounds=(0.30f0, 0.90f0), initial=0.60f0),

    # ── Longwave transmissivity (Frierson scheme) ─────────────────────────────
    # grad_scale validated in examples/trenberth_blowup_fix/trenberth_gradscale_fix.jl --
    # without it these gradients (raw magnitude ~400-600) fight cloud_albedo for the
    # shared grad_clip budget and cause a catastrophic blowup. Do not remove.
    ParamSpec(:tau0_equator,
        [:longwave_radiation, :transmissivity, :τ₀_equator];
        bounds=(2f0, 12f0), initial=6f0, grad_scale=0.04f0),
    ParamSpec(:tau0_pole,
        [:longwave_radiation, :transmissivity, :τ₀_pole];
        bounds=(0.3f0, 4f0), initial=1.5f0, grad_scale=0.25f0),
    ParamSpec(:fl,
        [:longwave_radiation, :transmissivity, :fₗ];
        bounds=(0.0f0, 0.5f0), initial=0.1f0, grad_scale=0.025f0),

    # ── Longwave emissivity ───────────────────────────────────────────────────
    ParamSpec(:emissivity_ocean,
        [:longwave_radiation, :radiative_transfer, :emissivity_ocean];
        bounds=(0.80f0, 1.00f0), initial=0.98f0, grad_scale=0.2f0),
    ParamSpec(:emissivity_land,
        [:longwave_radiation, :radiative_transfer, :emissivity_land];
        bounds=(0.80f0, 1.00f0), initial=0.98f0, grad_scale=0.5f0),

    # ── Land hydrology params REMOVED ─────────────────────────────────────────
    # infiltration_fraction, ocean_moisture, snow_melting_threshold all had
    # bit-exact-zero AD gradient in every run (dead weight under the online
    # single-timestep training method) -- see project_trenberth_lw_transmissivity_
    # gradscale_fix memory for root cause. Do not re-add without re-verifying.
]

println("$(length(param_specs)) trainable parameters defined.")

## 3. Configure the Loss Function

`TRENBERTH_LOSS` targets all 6 SW+LW fluxes. OLR and OSR receive full weight (1.0); surface fluxes receive lower weight (0.3–0.5) reflecting their larger observational uncertainty.

In [ ]:
# Relative-error weighting: weight_k = (osr_target/target_k)^2 -- an OBJECTIVE,
# formula-derived scheme. Replaces the old hand-tuned lrd=0.7 weight (retired --
# choosing per-flux weights by sweeping for a better score is the wrong approach,
# see feedback_training_weights memory). Validated: strictly dominates equal
# weighting (all=1.0) on all 6 fluxes at true 7-year equilibrium.
loss_config = LossConfig(
    [:osr, :sru, :srd, :olr, :lrd, :lru];
    targets = Dict(:osr => 101.9f0, :sru =>  23.0f0, :srd => 168.0f0,
                   :olr => 235.0f0, :lrd => 333.0f0, :lru => 398.0f0),
    weights = Dict(:osr => 1.00000f0, :sru => 19.62875f0, :srd => 0.36790f0,
                   :olr => 0.18802f0, :lrd => 0.09364f0,  :lru => 0.06555f0),
)

println("Loss configuration: $(length(loss_config.flux_keys))-flux MSE")
println()
@printf("  %-6s  %8s  %10s\n", "flux", "target", "weight")
println("  " * "-" ^ 28)
for k in loss_config.flux_keys
    @printf("  %-6s  %6.1f W/m²  %10.5f\n",
            k, loss_config.targets[k], loss_config.weights[k])
end

## 4. Quick Test

Verify all parameter paths are correct and Enzyme can differentiate through everything before the full run.

> **Note:** `calibrate!` warms up Enzyme automatically on the actual training model (`warmup_enzyme=true` by default). Expect *"Enzyme warmup complete in X s."* before the spinup on the first call. Pass `warmup_enzyme=false` on subsequent calls in the same session.

In [ ]:
result_test = calibrate!(
    param_specs,
    Optimisers.Adam(5f-3),
    loss_config,
    quick_test_config(),
)

println("Quick test complete: ", result_test.conv_info.stop_reason)

# Check gradient magnitudes — a zero gradient for any parameter is a red flag
println("\nGradient magnitudes (last batch):")
@printf("  %-28s  %12s\n", "parameter", "|mean grad|")
println("  " * "-" ^ 45)
for spec in result_test.param_specs
    g = result_test.history[Symbol("grad_", spec.name)]
    isempty(g) && continue
    @printf("  %-28s  %12.3e\n", spec.name, abs(g[end]))
end

**Interpreting gradient magnitudes:**  
- All gradients should be non-zero. A zero gradient means the parameter is disconnected from the loss through the AD graph — check the path or exclude the parameter.
- If one gradient is 100× larger than the others, consider increasing `grad_scale` for the weaker ones, or reducing it for the outlier.

---

## 5. Full Training Run

**⏱ Expected runtime: ~8–12 hours at T31.**

We use a slightly lower initial LR (`5f-3`) compared to the SW-only run because the 6-flux loss is more sensitive and can oscillate with a high LR.

In [ ]:
# batch_days=2 is the validated training window -- settled after a rigorous re-test
# (see project_trenberth_lw_transmissivity_gradscale_fix memory, "UPDATE 2026-08-06"):
# trained bd=2/5/10 ALL to real patience-based convergence under this loss and
# validated at true equilibrium. bd=2 beats both alternatives on 5 of 6 fluxes,
# roughly half the mean bias of either. An earlier screening sweep suggested longer
# windows fix lrd, but that was an early-training-snapshot artifact -- every longer
# window shows the same monotonic lrd drift once trained to real convergence, just
# later. Do not re-litigate this without a new reason to expect a different outcome.
#
# samples_per_batch=10 at batch_days=2: batch_steps=ceil(2*36)=72,
# steps_per_sample=72÷10=7, gcd(7,36)=1 -- clean, no diurnal aliasing.
#
# FORCE RETRAIN (2026-08-08): the previously saved result predates the grad_norm/
# clipped diagnostics added to TrainingResult.history. Back up the old converging
# result (loss 785→60, best_batch=342) and retrain from scratch so history carries
# the new fields and fig_loss shows the grad_norm-vs-clip panel. This run takes the
# full ~8-12h -- do not skip the backup step, it's the only copy of the validated
# pre-diagnostics result.
save_path = joinpath(@__DIR__, "output", "trenberth_full_result.jld2")
if isfile(save_path)
    backup_path = joinpath(@__DIR__, "output", "trenberth_full_result_prediagnostics.jld2")
    if !isfile(backup_path)
        cp(save_path, backup_path)
        println("Backed up existing result to: $backup_path")
    end
end

result = calibrate!(
    param_specs,
    Optimisers.Adam(5f-3),
    loss_config,
    TrainingConfig(
        spinup_days       = 180,
        batch_days        = 2.0,
        samples_per_batch = 10,
        max_batches       = 400,
        loss_threshold    = 1f-6,
        enable_lr_decay   = false,
        trunc             = 31,
        nlayers           = 8,
        daily_cycle       = true,   # required: physical diurnal cycle must stay on
    ),
)
mkpath(dirname(save_path))
save_result(result, save_path)
println("Result saved to: $save_path")

## 6. Inspect Results

In [ ]:
println(result)
println()
println("Convergence info:")
println("  stop_reason:        ", result.conv_info.stop_reason)
println("  best_smoothed_loss: ", round(result.conv_info.best_smoothed_loss, digits=2))
println("  total_batches:      ", result.conv_info.total_batches)
@printf("  total_time:         %.1f hours\n", result.conv_info.total_time / 3600)

In [ ]:
# Parameter table: initial → trained
@printf("\n%-28s  %10s  %10s  %10s\n", "parameter", "initial", "trained", "change")
println("-" ^ 65)
for spec in result.param_specs
    init    = isnothing(spec.initial) ? NaN32 : spec.initial
    trained = result.final_params[spec.name]
    @printf("%-28s  %10.4g  %10.4g  %+10.4g\n",
            spec.name, init, trained, trained - init)
end

In [ ]:
# Flux bias at end of training
@printf("\n%-6s  %8s  %8s  %8s\n", "flux", "target", "trained", "bias")
println("-" ^ 38)
for k in result.loss_config.flux_keys
    tgt  = result.loss_config.targets[k]
    val  = result.history[k][end]
    @printf("%-6s  %8.2f  %8.2f  %+8.2f\n", k, tgt, val, val - tgt)
end

## 7. Plot Training History

In [ ]:
figs = plot_training(result; save_dir=joinpath(@__DIR__, "output", "trenberth_full"))
figs.fig_loss

In [ ]:
figs.fig_flux

In [ ]:
figs.fig_params

In [ ]:
figs.fig_grads

## 8. Climate Validation

We compare the default model's equilibrium climatology against the trained one across all 6 target fluxes, precipitation, and the temperature profile.

**⏱ Expected runtime: ~1–3 hours at T31.**

In [ ]:
clm = run_climate_validation(result; n_years=7, stat_years=5, dt=Minute(20))

In [ ]:
# Full bias comparison table
targets = result.loss_config.targets

@printf("%-6s  %8s  %9s  %9s  %9s  %9s\n",
        "flux", "target",
        "def val", "def bias",
        "trn val", "trn bias")
println("-" ^ 60)
for k in result.loss_config.flux_keys
    tgt    = targets[k]
    d_val  = getproperty(clm.default, k)
    t_val  = getproperty(clm.trained, k)
    @printf("%-6s  %8.2f  %9.2f  %+9.2f  %9.2f  %+9.2f\n",
            k, tgt, d_val, d_val-tgt, t_val, t_val-tgt)
end
println()

# Also show precipitation (not in the loss — a held-out diagnostic)
println("Held-out diagnostics (not in loss):")
@printf("  Precipitation:  default = %.2f mm/day  trained = %.2f mm/day  (ERA5 ≈ 2.74)\n",
        clm.default.precip_total, clm.trained.precip_total)

In [ ]:
cfigs = plot_climate(clm;
    save_dir    = joinpath(@__DIR__, "output", "trenberth_full"),
    loss_config = result.loss_config,
)
cfigs.fig_rad

In [ ]:
cfigs.fig_lw

In [ ]:
cfigs.fig_precip

In [ ]:
cfigs.fig_summary

## 9. Interpretation and Next Steps

**Reading the summary plot:**  
Each bar shows the equilibrium bias (trained value − Trenberth target) for one flux. Bars shrinking toward zero from default (grey) to trained (blue) indicate successful calibration. Watch for biases that *grow* in fluxes not in the loss — these indicate compensatory parameter adjustments.

**Common issues and fixes:**

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Loss oscillates without decreasing | LR too high | Reduce Adam LR to `1f-3` |
| One flux improves, another degrades | Loss weights too uneven | Rebalance `TRENBERTH_LOSS` weights |
| Parameter hits its bound | Bounds too narrow | Widen the offending `ParamSpec` bounds |
| Precipitation degrades strongly | Convective params pulled too far | Add precipitation term to loss |
| Gradient ~0 for a parameter | Zero-gradient path | Exclude parameter or change its path |

**Continuing from this result:**  
Set `initial = result.final_params[spec.name]` in each `ParamSpec` and re-run with a lower LR to refine further. The sigmoid reparameterisation ensures bounds are still respected.